# HGT + Ensemble Extension — Privilege Propagation Graph

Trains and evaluates the third model (HGT, `model_hgt.py`) alongside the
existing GraphSAGE (primary) and GAT (baseline), then calibrates and
evaluates the GraphSAGE+HGT ensemble (`model_ensemble.py`).

**Reuses, unmodified:** `data_loader.py` (`PrivilegePropagationGraphLoader`,
splits), `utils.py` (`FocalLoss`, `evaluate`), `model_graphsage.py`,
`model_gat.py`. **New, from this extension:** `model_hgt.py`,
`node_importance.py`, `model_ensemble.py`, `train_scalable.py`.

No code here modifies the existing streaming pipeline, Neo4j schema, or
checkpoint format for GraphSAGE/GAT.


## 1. Install dependencies + detect GPU

In [ ]:
import subprocess, sys, torch

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

# Match the extra-index install PyG's docs recommend for the accelerated
# scatter/sparse kernels, keyed off whatever torch + CUDA Colab gives you.
TORCH_VERSION = torch.__version__.split("+")[0]
CUDA = "cu121" if torch.cuda.is_available() else "cpu"
pip_install("torch_geometric")
try:
    pip_install(
        "torch_scatter", "torch_sparse", "torch_cluster",
        "-f", f"https://data.pyg.org/whl/torch-{TORCH_VERSION}+{CUDA}.html",
    )
except Exception as e:
    print("Optional accelerated PyG kernels failed to install "
          f"({e}) — torch_geometric's pure-Python fallbacks still work, "
          "just slower. Safe to continue.")

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Upload the extension's new files + this repo's existing files

Upload `data_loader.py`, `utils.py`, `model_graphsage.py`, `model_gat.py`,
`explainability.py`, `cloudtrail_structural.csv` (existing) alongside
`model_hgt.py`, `node_importance.py`, `model_ensemble.py`,
`train_scalable.py` (new, from this extension) into the Colab
filesystem, or clone your repo — either way, everything below just
`import`s them by name.

In [ ]:
import sys
sys.path.insert(0, ".")  # adjust if files were uploaded/cloned elsewhere

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

from data_loader import PrivilegePropagationGraphLoader, NODE_FEATURE_SCHEMA, \
    stratified_edge_split, principal_disjoint_split, global_labels, flatten_mask_dict
from utils import FocalLoss, evaluate, print_confusion_matrix
from model_graphsage import GraphSAGEAnomalyDetector
from model_gat import GATAnomalyDetector
from model_hgt import HGTAnomalyDetector
from model_ensemble import EnsembleModel
from train_scalable import (
    ScalableTrainConfig, train_graphsage_minibatch, train_hgt,
    calibrate_ensemble_alpha, compare_n_way,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## 3. Load the graph (existing loader, unmodified) and split

In [ ]:
loader = PrivilegePropagationGraphLoader("cloudtrail_structural.csv")
data, meta = loader.load()

print("Node types:", data.node_types)
print("Populated edge triples:", len(meta["populated_triples"]))
print({t: data[t].edge_index.shape[1] for t in data.edge_types})

# Adjust to whichever split function matches how you trained the
# existing GraphSAGE/GAT checkpoints, so comparisons are apples-to-apples.
train_mask_dict, val_mask_dict, test_mask_dict = stratified_edge_split(data, meta)


## 4. Train GraphSAGE — mini-batch (LinkNeighborLoader)

`GraphSAGEAnomalyDetector` itself is byte-for-byte the same class
`train.py` already trains full-batch — this cell only demonstrates the
new scalable training PATH from `train_scalable.py`. On this dataset's
current scale, `train.py`'s existing full-batch loop is equally valid
and considerably simpler; use this cell when comparing against a larger
incrementally-updated graph.

In [ ]:
cfg = ScalableTrainConfig(epochs=60, batch_size=256, num_neighbors=[15, 10],
                           use_amp=torch.cuda.is_available(), device=str(device))

sage_model = GraphSAGEAnomalyDetector(
    node_feat_dims={nt: data[nt].x.shape[1] for nt in data.node_types},
    edge_types=meta["populated_triples"],
    edge_feat_dim=data[meta["populated_triples"][0]].edge_attr.shape[1],
)

sage_result = train_graphsage_minibatch(
    sage_model, data, data, train_mask_dict, val_mask_dict,
    meta["populated_triples"], cfg, model_name="GraphSAGE",
)
print("GraphSAGE best val F1:", sage_result["best_val_f1"])


## 5. Train HGT — HGTLoader over the node_importance-selected region

See `node_importance.py` for exactly which existing features feed the
ranking (`hop_count`, `privilege_gain`, `abnormal_path_frequency`,
`is_privilege_escalation_technique` aggregated onto nodes; native
`resource_sensitivity` / `distance_to_sensitive_resource` / degree) — no
new columns, no new graph construction.

In [ ]:
hgt_cfg = ScalableTrainConfig(epochs=60, batch_size=256, num_neighbors=[10, 10],
                               use_amp=torch.cuda.is_available(), device=str(device),
                               top_frac=0.2, min_per_type=1)

hgt_model = HGTAnomalyDetector(
    node_feat_dims={nt: data[nt].x.shape[1] for nt in data.node_types},
    edge_types=meta["populated_triples"],
    edge_feat_dim=data[meta["populated_triples"][0]].edge_attr.shape[1],
    heads=4, hidden_dim=128, num_hgt_layers=2,
)

hgt_result = train_hgt(
    hgt_model, data, data, train_mask_dict, val_mask_dict,
    meta["populated_triples"], NODE_FEATURE_SCHEMA, hgt_cfg, model_name="HGT",
)
print("HGT best val F1:", hgt_result["best_val_f1"])


## 6. GAT baseline (existing model, evaluated for comparison — not modified)

In [ ]:
# If you already have a trained GAT checkpoint from train.py, load it
# instead of retraining here — this cell trains one from scratch only if
# you don't. Uses train.py's own conventions (see that file) rather than
# train_scalable.py's, since GAT isn't part of this extension's scope.
gat_model = GATAnomalyDetector(
    node_feat_dims={nt: data[nt].x.shape[1] for nt in data.node_types},
    edge_types=meta["populated_triples"],
    edge_feat_dim=data[meta["populated_triples"][0]].edge_attr.shape[1],
).to(device)

opt = torch.optim.AdamW(gat_model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = FocalLoss()
for epoch in range(60):
    gat_model.train()
    opt.zero_grad()
    logits = gat_model(data.to(device))
    y = torch.as_tensor(global_labels(data), dtype=torch.float, device=device)
    loss = loss_fn(logits, y)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(gat_model.parameters(), 1.0)
    opt.step()
    if (epoch + 1) % 10 == 0:
        m = evaluate(gat_model, data, val_mask_dict)
        print(f"epoch {epoch+1}: loss={loss.item():.4f} val_f1={m['f1']:.4f}")


## 7. Calibrate and build the ensemble

In [ ]:
best_alpha, alpha_sweep = calibrate_ensemble_alpha(sage_model, hgt_model, data, val_mask_dict)
print("Chosen alpha:", best_alpha)

ensemble = EnsembleModel([("graphsage", sage_model, 1.0 - best_alpha), ("hgt", hgt_model, best_alpha)])
ensemble.eval()


## 8. Evaluate all four and compare

In [ ]:
sage_metrics = evaluate(sage_model, data, test_mask_dict, return_probs=True)
gat_metrics = evaluate(gat_model, data, test_mask_dict, return_probs=True)
hgt_metrics = evaluate(hgt_model, data, test_mask_dict, return_probs=True)
ensemble_metrics = evaluate(ensemble, data, test_mask_dict, return_probs=True)

compare_n_way({
    "GraphSAGE": sage_metrics, "GAT": gat_metrics,
    "HGT": hgt_metrics, "Ensemble": ensemble_metrics,
})
print_confusion_matrix(ensemble_metrics["confusion"])


## 9. Plots — training curves, ROC, PR, confusion matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve, ConfusionMatrixDisplay
import numpy as np

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

axes[0, 0].plot(sage_result["history"]["epoch"], sage_result["history"]["train_loss"], label="GraphSAGE")
axes[0, 0].plot(hgt_result["history"]["epoch"], hgt_result["history"]["train_loss"], label="HGT")
axes[0, 0].set_title("Training loss"); axes[0, 0].set_xlabel("epoch"); axes[0, 0].legend()

for name, m in [("GraphSAGE", sage_metrics), ("GAT", gat_metrics), ("HGT", hgt_metrics), ("Ensemble", ensemble_metrics)]:
    fpr, tpr, _ = roc_curve(m["labels"], m["probs"])
    axes[0, 1].plot(fpr, tpr, label=f"{name} (AUC={m['roc_auc']:.3f})")
axes[0, 1].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0, 1].set_title("ROC"); axes[0, 1].set_xlabel("FPR"); axes[0, 1].set_ylabel("TPR"); axes[0, 1].legend()

for name, m in [("GraphSAGE", sage_metrics), ("GAT", gat_metrics), ("HGT", hgt_metrics), ("Ensemble", ensemble_metrics)]:
    p, r, _ = precision_recall_curve(m["labels"], m["probs"])
    axes[0, 2].plot(r, p, label=name)
axes[0, 2].set_title("Precision-Recall"); axes[0, 2].set_xlabel("Recall"); axes[0, 2].set_ylabel("Precision"); axes[0, 2].legend()

ConfusionMatrixDisplay(np.array(ensemble_metrics["confusion"]), display_labels=["normal", "attack"]).plot(ax=axes[1, 0])
axes[1, 0].set_title("Ensemble confusion matrix")

names = ["GraphSAGE", "GAT", "HGT", "Ensemble"]
f1s = [sage_metrics["f1"], gat_metrics["f1"], hgt_metrics["f1"], ensemble_metrics["f1"]]
axes[1, 1].bar(names, f1s); axes[1, 1].set_title("Test F1 by model"); axes[1, 1].set_ylim(0, 1)

axes[1, 2].plot(list(alpha_sweep.keys()), [v["f1"] for v in alpha_sweep.values()], marker="o")
axes[1, 2].set_title("Ensemble alpha sweep (val F1)"); axes[1, 2].set_xlabel("alpha (HGT weight)")

plt.tight_layout()
plt.savefig("hgt_ensemble_results.png", dpi=150)
plt.show()


## 10. Save checkpoints + export metrics

`GraphSAGE`/`GAT` checkpoints are untouched bare `state_dict()`s, exactly
like `train.py` already produces (`train_graphsage_minibatch` follows
that same convention). `HGT` and `Ensemble` get their own wrapped
checkpoints in the `{state_dict, model_args}` sidecar shape
`infer.py`'s `load_model_from_checkpoint` already expects — self-
contained for the ensemble (embeds both components), per the design
notes on why paths-to-other-checkpoints was avoided.

In [ ]:
import json, os
os.makedirs("checkpoints", exist_ok=True)

common_edge_types = meta["populated_triples"]
edge_feat_dim = data[common_edge_types[0]].edge_attr.shape[1]
node_feat_dims = {nt: data[nt].x.shape[1] for nt in data.node_types}
fit_artifacts = getattr(loader, "fit_artifacts", {
    "node_scalers": getattr(loader, "node_scalers", {}),
    "edge_scaler": getattr(loader, "edge_scaler", None),
    "label_encoders": getattr(loader, "label_encoders", {}),
})

hgt_args = {
    "model_type": "hgt", "node_feat_dims": node_feat_dims,
    "edge_types": common_edge_types, "edge_feat_dim": edge_feat_dim,
    "hidden_dim": 128, "heads": 4, "num_hgt_layers": 2,
}
torch.save({"state_dict": hgt_model.state_dict(), "model_args": hgt_args,
            "fit_artifacts": fit_artifacts}, "checkpoints/best_HGT_wrapped.pt")

sage_args = {
    "model_type": "graphsage", "node_feat_dims": node_feat_dims,
    "edge_types": common_edge_types, "edge_feat_dim": edge_feat_dim,
    "hidden_dim": 128, "num_sage_layers": 2,
}
ensemble_args = {
    "model_type": "ensemble",
    "ensemble": {"components": [
        {"name": "graphsage", "model_type": "graphsage", "weight": 1.0 - best_alpha,
         "state_dict": sage_model.state_dict(), "model_args": sage_args},
        {"name": "hgt", "model_type": "hgt", "weight": best_alpha,
         "state_dict": hgt_model.state_dict(), "model_args": hgt_args},
    ]},
    # kept at top level too, purely for the ensemble-branch log line in
    # infer.py's load_model_from_checkpoint — not read for construction
    "edge_types": common_edge_types, "edge_feat_dim": edge_feat_dim,
}
torch.save({"model_args": ensemble_args, "fit_artifacts": fit_artifacts},
           "checkpoints/best_Ensemble_wrapped.pt")

with open("checkpoints/metrics.json", "w") as f:
    json.dump({
        "GraphSAGE": {k: v for k, v in sage_metrics.items() if k not in ("probs", "labels", "report")},
        "GAT": {k: v for k, v in gat_metrics.items() if k not in ("probs", "labels", "report")},
        "HGT": {k: v for k, v in hgt_metrics.items() if k not in ("probs", "labels", "report")},
        "Ensemble": {k: v for k, v in ensemble_metrics.items() if k not in ("probs", "labels", "report")},
        "best_alpha": best_alpha,
    }, f, indent=2)

print("Saved checkpoints/best_HGT_wrapped.pt, checkpoints/best_Ensemble_wrapped.pt, checkpoints/metrics.json")
print("To use in infer.py: python infer.py --checkpoint checkpoints/best_Ensemble_wrapped.pt ...")
